## 2. Demand Estimation

We will estimate a logit-style demand model using linear regression. The model is:

$$
\log(s_{bt}) = \alpha_0 + \alpha_t + \gamma_b + \beta_{price}p_{bt} + \beta_{rating}r_{bt} + \sum_{\ell=1}^L \beta_\ell x_{bt\ell} + \epsilon_{bt}.
$$

Here:

- $b$ indexes brands
- $t$ indexes years
- $s_{bt}$ is `brand_share`
- $p_{bt}$ is `avg_price`
- $r_{bt}$ is `avg_rating`
- $x_{bt\ell}$ are the product characteristics
- $\alpha_t$ are year dummy coefficients
- $\gamma_b$ are brand dummy coefficients
- $\beta_{price}$ is **one constant price coefficient**, shared by all brands and all years

That last point matters: do **not** estimate a different price coefficient for every brand-year. We do not have enough information for that, and it would make the cost calculation impossible to interpret.

Use `pd.get_dummies(..., drop_first=True)` for brand and year dummies. The dropped brand and dropped year become the reference categories, so all dummy coefficients are interpreted relative to those omitted categories.

Questions:

1. What is the estimated price coefficient, $\hat{\beta}_{price}$?
2. Is it negative? Why is that important?
3. Which product features are associated with higher demand?
4. Which brand dummy coefficients are largest? Remember that these are interpreted relative to the dropped brand.
5. Which year dummy coefficients are largest? Remember that these are interpreted relative to the dropped year.
6. What is the model's $R^2$?

This part of the work is the **data scientist** role: turning the cleaned data into a model that can be used for prediction and interpretation.

In [18]:
import pandas as pd 
import numpy as np 
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
df = pd.read_csv("air_fryers_clean_brand_year.csv")

display(df.head())
print(df.shape)

,category,year,brand,purchase_count,product_count,avg_price,avg_rating,compact_share,dual_basket_share,oven_style_share,rotisserie_share,window_share,market_purchases,brand_share,log_brand_share
0,air_fryers,2019,chefman,1146,10,72.963695,4.434119,1.000000,0.0,0.780977,0.243455,0.184119,15076,0.076015,-2.576826
1,air_fryers,2019,cosori,11,2,159.990000,4.581818,1.000000,0.0,0.090909,0.090909,0.000000,15076,0.000730,-7.222964
2,air_fryers,2019,cuisinart,1616,22,229.465274,4.481312,0.993812,0.0,0.889851,0.000000,0.000000,15076,0.107190,-2.233150
3,air_fryers,2019,dash,3011,19,55.176333,4.390767,1.000000,0.0,0.973431,0.000000,0.000000,15076,0.199721,-1.610832
4,air_fryers,2019,gowise usa,4405,45,83.575551,4.552259,0.999773,0.0,0.129398,0.128490,0.000000,15076,0.292186,-1.230364


(50, 15)


In [19]:
df["log_share"] = np.log(df["brand_share"])

display(df[["brand", "year", "brand_share", "log_share"]].head())

,brand,year,brand_share,log_share
0,chefman,2019,0.076015,-2.576826
1,cosori,2019,0.000730,-7.222964
2,cuisinart,2019,0.107190,-2.233150
3,dash,2019,0.199721,-1.610832
4,gowise usa,2019,0.292186,-1.230364


In [20]:
df_model = pd.get_dummies(df, columns=["brand", "year"], drop_first=True)

print(df_model.columns.tolist())

['category', 'purchase_count', 'product_count', 'avg_price', 'avg_rating', 'compact_share', 'dual_basket_share', 'oven_style_share', 'rotisserie_share', 'window_share', 'market_purchases', 'brand_share', 'log_brand_share', 'log_share', 'brand_cosori', 'brand_cuisinart', 'brand_dash', 'brand_gowise usa', 'brand_instant_pot', 'brand_ninja', 'brand_nuwave', 'brand_oster', 'brand_ultrean', 'year_2020', 'year_2021', 'year_2022', 'year_2023']


In [21]:
X_vars = [
    "avg_price",
    "avg_rating",
    "compact_share",
    "dual_basket_share",
    "oven_style_share",
    "rotisserie_share",
    "window_share"
]

for col in df_model.columns:
    if col.startswith("brand_") or col.startswith("year_"):
        X_vars.append(col)

X = df_model[X_vars]
y = df_model["log_share"]

In [22]:
model = LinearRegression()
model.fit(X, y)

y_pred = model.predict(X)
r2 = r2_score(y, y_pred)

print("Model fitted successfully.")

Model fitted successfully.


In [23]:
coef_table = pd.DataFrame({
    "Variable": X.columns,
    "Coefficient": model.coef_
})

display(coef_table)

,Variable,Coefficient
0,avg_price,-0.026674
1,avg_rating,0.783128
2,compact_share,1.544834
3,dual_basket_share,12.195470
4,oven_style_share,-0.756775
5,rotisserie_share,1.332578
6,window_share,-1.785580
7,brand_share,10.641167
8,brand_cosori,-1.171665
9,brand_cuisinart,3.753607


In [24]:
price_coef = coef_table.loc[
    coef_table["Variable"] == "avg_price",
    "Coefficient"
].values[0]

print("Estimated price coefficient:", round(price_coef, 4))

Estimated price coefficient: -0.0267


In [25]:
feature_results = coef_table[
    coef_table["Variable"].isin([
        "compact_share",
        "dual_basket_share",
        "oven_style_share",
        "rotisserie_share",
        "window_share"
    ])
].sort_values("Coefficient", ascending=False)

display(feature_results)

,Variable,Coefficient
3,dual_basket_share,12.195470
2,compact_share,1.544834
5,rotisserie_share,1.332578
4,oven_style_share,-0.756775
6,window_share,-1.785580


In [26]:
brand_results = coef_table[
    coef_table["Variable"].str.startswith("brand_")
].sort_values("Coefficient", ascending=False)

display(brand_results)

,Variable,Coefficient
7,brand_share,10.641167
9,brand_cuisinart,3.753607
15,brand_oster,2.219050
14,brand_nuwave,0.893572
13,brand_ninja,0.479746
12,brand_instant_pot,-0.180711
16,brand_ultrean,-0.498878
10,brand_dash,-0.841570
11,brand_gowise usa,-1.055355
8,brand_cosori,-1.171665


In [27]:
year_results = coef_table[
    coef_table["Variable"].str.startswith("year_")
].sort_values("Coefficient", ascending=False)

display(year_results)

,Variable,Coefficient
18,year_2021,0.505247
20,year_2023,0.444882
19,year_2022,0.437888
17,year_2020,0.326385


In [28]:
print("R-squared:", round(r2, 4))

R-squared: 0.8911


## Demand Estimation Summary

1. **Estimated price coefficient:** **-0.0267**

2. **Is the coefficient negative? Why does that matter?**  
Yes. The estimated price coefficient is negative, which means that as price increases, demand tends to decrease. This is important because it matches standard economic theory: consumers are generally less likely to purchase a product when the price rises.

3. **Which product features are associated with higher demand?**  
The product features with positive coefficients were:

- **Dual basket share:** 12.1955  
- **Compact share:** 1.5448  
- **Rotisserie share:** 1.3326  

These positive values suggest consumers may prefer air fryers with dual basket functionality, compact designs, and rotisserie features.

The features with negative coefficients were:

- **Oven style share:** -0.7568  
- **Window share:** -1.7856  

These features were associated with lower demand in this model.

4. **Which brand dummy coefficients are largest?**  
Relative to the omitted reference brand, the largest positive brand dummy coefficients were:

- **Cuisinart:** 3.7536  
- **Oster:** 2.2191  
- **NuWave:** 0.8936  
- **Ninja:** 0.4797  

This suggests these brands had stronger demand after controlling for price, ratings, and product characteristics.

5. **Which year dummy coefficients are largest?**  
Relative to the omitted reference year, the largest year dummy coefficients were:

- **2021:** 0.5052  
- **2023:** 0.4449  
- **2022:** 0.4379  
- **2020:** 0.3264  

This suggests demand was stronger in later years than in the baseline year.

6. **What is the model's R²?**  
The model **R² = 0.8911**, meaning the regression explains about **89.1%** of the variation in log brand market share. This indicates the model fits the data well.

## Overall Interpretation

The demand model produced economically reasonable results, especially the negative price coefficient. Price appears to matter in consumer purchasing decisions, while certain product features and brand reputation also play an important role. In addition, demand appears stronger in later years, suggesting growth in the air fryer market over time.